In [1]:
import pandas as pd
import numpy as np
import nltk
import re

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [4]:
df = pd.read_csv("/content/job_descriptions.csv", engine='python', on_bad_lines='warn')

print(df.head())
print(df.columns)

/tmp/ipykernel_5950/3822726055.py:1: ParserWarning: Skipping line 43711: unexpected end of data

  df = pd.read_csv("/content/job_descriptions.csv", engine='python', on_bad_lines='warn')


             Job Id     Experience Qualifications Salary Range    location  \
0  1089843540111562  5 to 15 Years         M.Tech    $59K-$99K     Douglas   
1   398454096642776  2 to 12 Years            BCA   $56K-$116K    Ashgabat   
2   481640072963533  0 to 12 Years            PhD   $61K-$104K       Macao   
3   688192671473044  4 to 11 Years            PhD    $65K-$91K  Porto-Novo   
4   117057806156508  1 to 12 Years            MBA    $64K-$87K    Santiago   

            Country  latitude  longitude  Work Type  Company Size  ...  \
0       Isle of Man   54.2361    -4.5481     Intern         26801  ...   
1      Turkmenistan   38.9697    59.5563     Intern        100340  ...   
2  Macao SAR, China   22.1987   113.5439  Temporary         84525  ...   
3             Benin    9.3077     2.3158  Full-Time        129896  ...   
4             Chile  -35.6751   -71.5429     Intern         53944  ...   

                 Contact                     Job Title  \
0   001-381-930-7517x737  Di

In [7]:
df['content'] = (
    df['Job Title'].fillna('') + " " +
    df['Role'].fillna('') + " " +
    df['Job Description'].fillna('') + " " +
    df['skills'].fillna('') + " " +
    df['Responsibilities'].fillna('')
)
print(df[['Job Title','content']].head())

                      Job Title  \
0  Digital Marketing Specialist   
1                 Web Developer   
2            Operations Manager   
3              Network Engineer   
4                 Event Manager   

                                             content  
0  Digital Marketing Specialist Social Media Mana...  
1  Web Developer Frontend Web Developer Frontend ...  
2  Operations Manager Quality Control Manager Qua...  
3  Network Engineer Wireless Network Engineer Wir...  
4  Event Manager Conference Manager A Conference ...  


In [26]:

stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()

    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

df['cleaned_content'] = df['content'].apply(preprocess)

print(df[['content', 'cleaned_content']].head())

                                             content  \
0  Digital Marketing Specialist Social Media Mana...   
1  Web Developer Frontend Web Developer Frontend ...   
2  Operations Manager Quality Control Manager Qua...   
3  Network Engineer Wireless Network Engineer Wir...   
4  Event Manager Conference Manager A Conference ...   

                                     cleaned_content  
0  digital marketing specialist social media mana...  
1  web developer frontend web developer frontend ...  
2  operations manager quality control manager qua...  
3  network engineer wireless network engineer wir...  
4  event manager conference manager conference ma...  


In [12]:
#Task 2: TF-IDF

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['cleaned_content'])

In [10]:
query = input("Enter job search query: ")

Enter job search query: Ai job


In [13]:
query_clean = preprocess(query)

query_vector = vectorizer.transform([query_clean])

In [14]:
similarity_scores = cosine_similarity(query_vector, tfidf_matrix)

In [15]:
scores = similarity_scores.flatten()

top_indices = scores.argsort()[-5:][::-1]

In [17]:
print("Top 5 Matching Jobs:\n")

for i in top_indices:
    print("Job Title:", df.iloc[i]['Job Title'])
    print("Company Name:", df.iloc[i]['Company'])
    print("Score:", scores[i])
    print("-"*50)

Top 5 Matching Jobs:

Job Title: HR Generalist
Company Name: Meta Platforms
Score: 0.121805145190331
--------------------------------------------------
Job Title: HR Generalist
Company Name: Fidelity National Information Services
Score: 0.121805145190331
--------------------------------------------------
Job Title: HR Generalist
Company Name: Sprint Corporation
Score: 0.121805145190331
--------------------------------------------------
Job Title: HR Generalist
Company Name: Wesfarmers Limited
Score: 0.121805145190331
--------------------------------------------------
Job Title: HR Generalist
Company Name: Petronet LNG
Score: 0.121805145190331
--------------------------------------------------


In [ ]:
#Task 3: Build Basic Search System

In [20]:
def basic_search(query):
    query_clean = preprocess(query)

    query_vector = vectorizer.transform([query_clean])

    similarity = cosine_similarity(query_vector, tfidf_matrix)

    scores = similarity.flatten()

    top_indices = scores.argsort()[-5:][::-1]

    print("\nTop 5 Search Results:\n")

    for i in top_indices:
        print("Job Title:", df.iloc[i]['Job Title'])
        print("Company Name:", df.iloc[i]['Company'])
        print("Location:", df.iloc[i]['location'])
        print("Score:", round(scores[i],4))
        print("-"*50)

In [ ]:
#Task 4: Query Expansion using WordNet

In [24]:
def expand_query(query):
    expanded_terms = []

    for word in query.split():
        expanded_terms.append(word)

        synsets = wordnet.synsets(word)

        for syn in synsets[:2]:
            for lemma in syn.lemmas()[:2]:
                synonym = lemma.name().replace("_", " ")
                expanded_terms.append(synonym)

    expanded_query = " ".join(set(expanded_terms))

    return expanded_query
def search_with_expansion(query):
    expanded_query = expand_query(query)

    print("Original Query:", query)
    print("Expanded Query:", expanded_query)

    query_clean = preprocess(expanded_query)

    query_vector = vectorizer.transform([query_clean])

    similarity = cosine_similarity(query_vector, tfidf_matrix)

    scores = similarity.flatten()

    top_indices = scores.argsort()[-5:][::-1]

    print("\nTop 5 Results After Query Expansion:\n")

    for i in top_indices:
        print("Job Title:", df.iloc[i]['Job Title'])
        print("Company Name:", df.iloc[i]['Company'])
        print("Location:", df.iloc[i]['location'])
        print("Score:", round(scores[i],4))
        print("-"*50)

In [22]:
#Task 5 Result

In [25]:
print("WITHOUT QUERY EXPANSION")
basic_search("AI job")

print("\nWITH QUERY EXPANSION")
search_with_expansion("AI job")

WITHOUT QUERY EXPANSION

Top 5 Search Results:

Job Title: HR Generalist
Company Name: Meta Platforms
Location: Caracas
Score: 0.1218
--------------------------------------------------
Job Title: HR Generalist
Company Name: Fidelity National Information Services
Location: Ljubljana
Score: 0.1218
--------------------------------------------------
Job Title: HR Generalist
Company Name: Sprint Corporation
Location: Dili
Score: 0.1218
--------------------------------------------------
Job Title: HR Generalist
Company Name: Wesfarmers Limited
Location: Mbabane
Score: 0.1218
--------------------------------------------------
Job Title: HR Generalist
Company Name: Petronet LNG
Location: Gaza
Score: 0.1218
--------------------------------------------------

WITH QUERY EXPANSION
Original Query: AI job
Expanded Query: occupation business artificial intelligence Army Intelligence task job AI

Top 5 Results After Query Expansion:

Job Title: Data Analyst
Company Name: Comcast
Location: Dushanbe
Sc